# Bert-VITS2 Training — Indian English (LIMMITS'24)

**From scratch** with DeBERTa BERT conditioning for natural prosody.

| Component | License | Details |
|-----------|---------|--------|
| Our VITS2 code | Ours | Enterprise safe |
| DeBERTa-v3-base | MIT (Microsoft) | Enterprise safe |
| LIMMITS'24 data | CC-BY 4.0 | 80 hrs, 2 speakers, studio quality |

| Step | What | Time |
|------|------|------|
| 1 | Setup + install | ~5 min |
| 2 | Download LIMMITS'24 (21 GB) | ~20-30 min |
| 3 | Process audio | ~30-60 min |
| 4 | Train (checkpoints every 2K steps) | 12-24 hrs |
| 5 | Generate podcast from any checkpoint | ~5 min |

**Bulletproof checkpointing:** Every 2K steps saved to Google Drive (Colab) or persistent storage (RunPod). Resume from any crash.

Works on: **Colab H100**, **Colab A100**, **RunPod RTX 4090**

## Step 1: Setup

In [ ]:
import os, sys, shutil, glob

# --- Detect environment ---
IS_COLAB = os.path.exists('/content') and 'COLAB_RELEASE_TAG' in os.environ
BASE = '/content' if IS_COLAB else '/workspace'
print(f"Environment: {'Colab' if IS_COLAB else 'RunPod/Other'}")
print(f"Base: {BASE}")

# --- GPU ---
import torch
assert torch.cuda.is_available(), "No GPU!"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME} ({GPU_MEM:.0f} GB)")

# Auto-select batch size based on GPU
if 'H100' in GPU_NAME:
    BATCH_SIZE = 32
elif 'A100' in GPU_NAME:
    BATCH_SIZE = 24
elif '4090' in GPU_NAME:
    BATCH_SIZE = 16
else:
    BATCH_SIZE = 12
print(f"Auto batch size: {BATCH_SIZE}")

# --- HuggingFace Login ---
try:
    if IS_COLAB:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    !pip install -q huggingface_hub
    from huggingface_hub import login
    login()
else:
    from huggingface_hub import HfApi
    try:
        print(f"HF user: {HfApi().whoami()['name']}")
    except Exception:
        from huggingface_hub import login
        login()

# --- Backup directory (survives crashes) ---
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_DIR = '/content/drive/MyDrive/bert_vits2_indian_tts'
else:
    BACKUP_DIR = f'{BASE}/backup_bert_vits2'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backup dir: {BACKUP_DIR}")

# --- Clone repo + install ---
os.chdir(BASE)
REPO_DIR = f'{BASE}/indian_tts'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/seetha0712/text2speech_1.git {REPO_DIR}
os.chdir(REPO_DIR)
!git checkout claude/custom-indian-tts-model-TUAjJ -q
!git pull origin claude/custom-indian-tts-model-TUAjJ -q

# Install deps
!apt-get -qq install -y espeak-ng ffmpeg sox libsox-dev > /dev/null 2>&1
!pip install -q -r requirements.txt 2>&1 | tail -1
!pip install -q -e . 2>&1 | tail -1
# BERT dependency
!pip install -q 'transformers>=4.45,<4.50' sentencepiece 2>&1 | tail -1

print(f"\nStep 1 complete!")

## Step 2: Download LIMMITS'24 Data (80 hrs Indian English)

Downloads ~21 GB of studio-quality Indian English speech (40 hrs male + 40 hrs female).

This takes ~20-30 min. The download is resumable — if it crashes, re-run this cell.

In [ ]:
DATA_DIR = f'{BASE}/data_limmits'

# Check if data already processed
if os.path.exists(f'{DATA_DIR}/train.txt'):
    with open(f'{DATA_DIR}/train.txt') as f:
        n = sum(1 for l in f if l.strip() and not l.startswith('#'))
    print(f"Data already ready: {n} training samples")
    print("Skip this cell if you don't want to re-download.")
else:
    os.chdir(REPO_DIR)
    !python -m indian_tts.data.download_limmits --output {DATA_DIR} --target-sr 22050

## Step 3: Configure Training

In [ ]:
import yaml

# Load Bert-VITS2 config
with open(f'{REPO_DIR}/configs/bert_vits2_limmits.yaml') as f:
    config = yaml.safe_load(f)

# Set paths
config['data']['training_files'] = f'{DATA_DIR}/train.txt'
config['data']['validation_files'] = f'{DATA_DIR}/val.txt'
config['paths']['output_dir'] = f'{BASE}/outputs_bert_vits2'
config['paths']['checkpoint_dir'] = f'{BASE}/outputs_bert_vits2/checkpoints'
config['paths']['log_dir'] = f'{BASE}/outputs_bert_vits2/logs'
config['paths']['backup_dir'] = BACKUP_DIR  # Bulletproof backup

# GPU-specific batch size
config['training']['batch_size'] = BATCH_SIZE

# Save config
CONFIG_PATH = f'{BASE}/bert_vits2_config.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config saved: {CONFIG_PATH}")
print(f"  BERT: {config['model']['use_bert']}")
print(f"  BERT model: {config['model']['bert_model_name']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Max steps: {config['training']['max_steps']}")
print(f"  Checkpoint every: {config['training']['save_every_n_steps']} steps")
print(f"  Backup to: {BACKUP_DIR}")

# Verify data
for fname in ['train.txt', 'val.txt']:
    path = f'{DATA_DIR}/{fname}'
    if os.path.exists(path):
        with open(path) as fh:
            n = sum(1 for l in fh if l.strip() and not l.startswith('#'))
        print(f"  {fname}: {n} samples")
    else:
        print(f"  ERROR: {path} not found!")

# Also backup the config
shutil.copy2(CONFIG_PATH, BACKUP_DIR)
print(f"\nConfig backed up to Drive.")

## Step 4: Train

**Checkpoints saved every 2K steps** to both local and Google Drive.

If session crashes: re-run Steps 1 and 3, then this cell. It auto-resumes from the latest checkpoint.

**Estimated time:**
- H100: ~12 hrs for 200K steps
- A100: ~24 hrs for 200K steps
- RTX 4090: ~30 hrs for 200K steps

In [ ]:
# Find latest checkpoint (local or backup)
import glob

CKPT_DIR = f'{BASE}/outputs_bert_vits2/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

local_ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint_step_*.pt'))
backup_ckpts = sorted(glob.glob(f'{BACKUP_DIR}/checkpoint_step_*.pt'))

# Restore from backup if local is empty
if not local_ckpts and backup_ckpts:
    latest_backup = backup_ckpts[-1]
    print(f"Restoring from backup: {os.path.basename(latest_backup)}")
    shutil.copy2(latest_backup, CKPT_DIR)
    local_ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint_step_*.pt'))

resume_flag = f'--resume {local_ckpts[-1]}' if local_ckpts else ''
if local_ckpts:
    print(f"Resuming from: {local_ckpts[-1]}")
else:
    print("Starting fresh training.")

os.chdir(REPO_DIR)
print(f"\nTraining with Bert-VITS2 on {GPU_NAME}...")
print(f"Checkpoints saved to: {CKPT_DIR}")
print(f"Backed up to: {BACKUP_DIR}")
print(f"\nStarting...\n")

!python -m indian_tts.train --config {CONFIG_PATH} {resume_flag}

## Step 5: Generate Podcast from Any Checkpoint

Use this to listen to the model at any training stage.

In [ ]:
import IPython.display as ipd
import numpy as np
import soundfile as sf
import sys

# Force module reload (in case of kernel restart)
if 'indian_tts' in sys.modules:
    for key in list(sys.modules.keys()):
        if key.startswith('indian_tts'):
            del sys.modules[key]
sys.path.insert(0, f'{REPO_DIR}/src')

# Find checkpoints
CKPT_DIR = f'{BASE}/outputs_bert_vits2/checkpoints'
ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint_*.pt') + glob.glob(f'{BACKUP_DIR}/checkpoint_*.pt'))

if not ckpts:
    print("No checkpoints found. Train first (Step 4).")
else:
    print("Available checkpoints:")
    for c in ckpts[-5:]:
        size_mb = os.path.getsize(c) / 1e6
        print(f"  {os.path.basename(c)} ({size_mb:.0f} MB)")
    
    # Use latest
    CHECKPOINT = ckpts[-1]
    print(f"\nUsing: {CHECKPOINT}")

In [ ]:
# Generate podcast
os.chdir(REPO_DIR)
PODCAST_DIR = f'{BASE}/outputs_bert_vits2/podcast'

!python -m indian_tts.podcast_demo \
    --checkpoint {CHECKPOINT} \
    --output {PODCAST_DIR}

# Listen
podcast_path = f'{PODCAST_DIR}/podcast_full.wav'
if os.path.exists(podcast_path):
    print("\nFull podcast:")
    ipd.display(ipd.Audio(podcast_path))
    
    # Backup
    step_name = os.path.basename(CHECKPOINT).replace('.pt', '')
    shutil.copy2(podcast_path, os.path.join(BACKUP_DIR, f'podcast_{step_name}.wav'))
    print(f"Saved to backup: podcast_{step_name}.wav")
else:
    print("Podcast generation failed. Check errors above.")

## Quick Test: Single Sentence

In [ ]:
from indian_tts.inference import IndianTTS

tts = IndianTTS(CHECKPOINT)

test_text = "Good morning everyone. Today we will discuss the quarterly results."

for voice in ['male', 'female']:
    audio = tts.synthesize(test_text, voice=voice, speed=1.1)
    print(f"\n[{voice.upper()}] {test_text}")
    ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

## Resume After Crash

If your session crashes:
1. Start a new session
2. Run Step 1 (setup)
3. Run Step 3 (config) — data is already downloaded if on persistent storage
4. Run Step 4 (train) — auto-resumes from latest Drive backup

You lose at most 2K steps (~15 min on H100).